# Food vs Non-Food Image Classification with CNN

**Python | TensorFlow | Keras | Computer Vision | Deep Learning**

This project implements a **Convolutional Neural Network (CNN)** to classify images into two categories:

- Food
- Non-Food

The model is developed using **TensorFlow/Keras** and trained on the **Food-5K dataset**.  
The workflow includes dataset loading, preprocessing, data augmentation, CNN model design, training diagnostics, and performance evaluation.

The objective of this project is not only to build a binary image classifier, but also to analyze how architectural choices and regularization techniques such as **dropout** and **data augmentation** influence model generalization.

---

# Dataset Description

This project uses the **Food-5K dataset**, which contains approximately **5,000 images** divided into two classes:

- Food  
- Non-Food  

The dataset is organized into three subsets:

- **Training set**
- **Validation set**
- **Evaluation (test) set**

Images are loaded using TensorFlow's `image_dataset_from_directory()` utility and resized to **256 × 256** resolution for model training.

**Dataset Source**

Kaggle – Food-5K Image Dataset  
https://www.kaggle.com/trolukovich/food5k-image-dataset


In [ ]:
# Dataset Path Configuration
# Define the root directory for the Food-5K dataset

import pathlib

dataset_path = pathlib.Path("Food-5K")

# The secret dataset directory contains two subfolders:
# 'food' and 'non-food'
secret_dataset_path = pathlib.Path("Secret")


## CNN Model Architecture

The classifier is built using a custom **Convolutional Neural Network (CNN)** designed for binary image classification.  
The network learns spatial features through stacked convolutional layers and progressively reduces spatial dimensions using pooling operations. The extracted feature representations are then passed to dense layers for final classification.

### Architecture Overview

The model follows a standard convolutional feature-extraction pipeline:

- **Input layer:** 256 × 256 × 3 RGB image  
- **Convolution Block 1:** 32 filters  
- **Convolution Block 2:** 64 filters  
- **Convolution Block 3:** 128 filters  
- **MaxPooling layers** applied after each convolution block  
- **Global Average Pooling** for dimensionality reduction  
- **Dense hidden layer** with ReLU activation  
- **Output layer** with Sigmoid activation for binary classification

The model is trained using **binary cross-entropy loss**, which is appropriate for two-class classification problems.

---

## Experimental Setup

To evaluate the CNN model and ensure robust performance, the following experimental setup is used:

- Training images resized to **256 × 256 resolution**
- **Data augmentation** applied to improve model generalization
- Training and validation **accuracy/loss monitoring**
- Evaluation of **dropout regularization** effects
- Comparison of multiple CNN configurations
- Selection of the **best model based on validation performance**


In [ ]:
# -------------------------------------------------------
# Dataset Directory Paths
# -------------------------------------------------------

from pathlib import Path

# Root directory of the Food-5K dataset
dataset_path = Path("Food-5K")

# Directory containing the hidden evaluation dataset
# (organized into 'food' and 'non-food' subfolders)
secret_dataset_path = Path("Secret")


In [ ]:
# -------------------------------------------------------
# Import Required Libraries
# -------------------------------------------------------

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import os

# -------------------------------------------------------
# Training Configuration (Hyperparameters)
# -------------------------------------------------------

IMAGE_SIZE = 256     # Input image resolution
BATCH_SIZE = 32      # Batch size used during training
SEED = 42            # Random seed for reproducibility
EPOCHS = 15          # Number of training epochs


In [ ]:
# -------------------------------------------------------
# Load Dataset
# -------------------------------------------------------

# Training dataset
training_dataset = keras.utils.image_dataset_from_directory(
    dataset_path / "training",
    labels="inferred",
    label_mode="binary",
    class_names=["food", "non_food"],
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    seed=SEED,
    shuffle=True
)

# Validation dataset
validation_dataset = keras.utils.image_dataset_from_directory(
    dataset_path / "validation",
    labels="inferred",
    label_mode="binary",
    class_names=["food", "non_food"],
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    seed=SEED,
    shuffle=False
)

# Test dataset
test_dataset = keras.utils.image_dataset_from_directory(
    dataset_path / "evaluation",
    labels="inferred",
    label_mode="binary",
    class_names=["food", "non_food"],
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    seed=SEED,
    shuffle=False
)

# -------------------------------------------------------
# Dataset Performance Optimization
# -------------------------------------------------------

training_dataset = training_dataset.cache().shuffle(1000)
validation_dataset = validation_dataset.cache()
test_dataset = test_dataset.cache()


In [ ]:
# -------------------------------------------------------
# Image Preprocessing: Preserve Aspect Ratio
# -------------------------------------------------------

def resize_with_pad(img, label):
    """
    Resize images while preserving the original aspect ratio.
    Padding is added when necessary to reach the target size.
    """
    img = tf.image.resize_with_pad(img, IMAGE_SIZE, IMAGE_SIZE)
    return img, label

training_dataset = training_dataset.map(resize_with_pad)
validation_dataset = validation_dataset.map(resize_with_pad)
test_dataset = test_dataset.map(resize_with_pad)


In [ ]:
# -------------------------------------------------------
# Data Augmentation
# -------------------------------------------------------

data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
        layers.RandomContrast(0.1),
    ],
    name="data_augmentation"
)


In [ ]:
# -------------------------------------------------------
# CNN Model Definition
# -------------------------------------------------------

def make_cnn(dropout=False, drop_rate=0.3):

    inputs = keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))

    # Data augmentation + normalization
    x = data_augmentation(inputs)
    x = layers.Rescaling(1./255)(x)

    # ----- Convolution Block 1 -----
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    if dropout:
        x = layers.Dropout(drop_rate)(x)

    # ----- Convolution Block 2 -----
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    if dropout:
        x = layers.Dropout(drop_rate)(x)

    # ----- Convolution Block 3 -----
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    if dropout:
        x = layers.Dropout(drop_rate)(x)

    # Global feature aggregation
    x = layers.GlobalAveragePooling2D()(x)

    if dropout:
        x = layers.Dropout(drop_rate)(x)

    # Fully connected layer
    x = layers.Dense(128, activation="relu")(x)

    if dropout:
        x = layers.Dropout(drop_rate)(x)

    # Output layer
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(
        inputs,
        outputs,
        name=f"small_cnn_{'dropout' if dropout else 'baseline'}"
    )

    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


# Create two models for comparison
model_A = make_cnn(dropout=False)           # Baseline CNN
model_B = make_cnn(dropout=True, drop_rate=0.4)  # CNN with dropout


## Experiment: Dropout Regularization

To study the effect of **dropout regularization** on model generalization, two CNN variants are trained and compared:

- **Model A:** Baseline CNN without dropout
- **Model B:** CNN with dropout regularization

The primary objective of this experiment is to evaluate whether dropout improves validation performance and reduces overfitting.

After training, the model with the **highest validation accuracy** is selected and saved as the final baseline model.


In [ ]:
# -------------------------------------------------------
# Train Both CNN Variants
# -------------------------------------------------------

history_A = model_A.fit(
    training_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS
)

history_B = model_B.fit(
    training_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS
)


In [ ]:
if max(history_B.history["val_accuracy"]) >= max(history_A.history["val_accuracy"]):
    model_B.save("food_nonfood_baseline.keras")
    print("Model B is Saved (with dropout)")
else:
    model_A.save("food_nonfood_baseline.keras")
    print("Model A is Saved (without dropout)")

In [ ]:
# -------------------------------------------------------
# Plot Training and Validation Curves
# -------------------------------------------------------

def plot_history(history, title):
    # Accuracy curve
    plt.figure()
    plt.plot(history.history["accuracy"], label="Training Accuracy")
    plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
    plt.title(f"{title} - Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)
    plt.show()

    # Loss curve
    plt.figure()
    plt.plot(history.history["loss"], label="Training Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.title(f"{title} - Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.show()


plot_history(history_A, "Model A (Without Dropout)")
plot_history(history_B, "Model B (With Dropout)")


In [ ]:
# -------------------------------------------------------
# Select the Best Model Based on Validation Accuracy
# -------------------------------------------------------

val_best_A = max(history_A.history["val_accuracy"])
val_best_B = max(history_B.history["val_accuracy"])

if val_best_B >= val_best_A:
    best_model = model_B
    final_model_name = "Model B (With Dropout)"
else:
    best_model = model_A
    final_model_name = "Model A (Without Dropout)"

final_path = "food_nonfood_baseline.keras"
best_model.save(final_path)

print(f"Best model saved: {final_model_name} -> {final_path}")


## Training Results

Training and validation **accuracy and loss curves** are analyzed to evaluate model convergence and generalization performance.

### Observations

- Training accuracy increases steadily during the learning process  
- Validation accuracy stabilizes after several epochs  
- Dropout regularization helps reduce overfitting by improving validation stability  

The model achieving the **highest validation accuracy** is selected and saved for later evaluation.

---

## Key Takeaways

- **Data augmentation** improves model robustness by introducing variability in training images  
- **Dropout regularization** helps prevent overfitting in convolutional networks  
- CNN architectures are effective at extracting **hierarchical spatial features** from image data  


## Model Evaluation

After training, the best-performing model is loaded from disk and evaluated on the **test dataset** to measure its generalization performance.

The evaluation process includes:

- Loading the saved CNN model
- Computing performance metrics on the test dataset
- Optionally evaluating on an additional dataset if available

This step verifies that the trained model performs consistently on unseen data.


In [ ]:
# -------------------------------------------------------
# Create Evaluation Dataset
# -------------------------------------------------------

def make_eval_dataset(root_dir: Path):

    dataset = keras.utils.image_dataset_from_directory(
        root_dir,
        labels="inferred",
        label_mode="binary",
        class_names=["food", "non_food"],
        image_size=(IMAGE_SIZE, IMAGE_SIZE),
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    def resize_with_pad(x, y):
        x = tf.image.resize_with_pad(x, IMAGE_SIZE, IMAGE_SIZE)
        return x, y

    return dataset.map(resize_with_pad)


# -------------------------------------------------------
# Load Saved Model
# -------------------------------------------------------

model = keras.models.load_model("food_nonfood_baseline.keras")


# -------------------------------------------------------
# Evaluate on Test Dataset
# -------------------------------------------------------

test_dataset = make_eval_dataset(dataset_path / "evaluation")

test_loss, test_acc = model.evaluate(test_dataset, verbose=0)

print(f"Food-5K Test Loss: {test_loss:.4f}")
print(f"Food-5K Test Accuracy: {test_acc * 100:.2f}%")


# -------------------------------------------------------
# Optional Evaluation on Additional Dataset
# -------------------------------------------------------

if secret_dataset_path.exists():

    secret_dataset = make_eval_dataset(secret_dataset_path)

    sec_loss, sec_acc = model.evaluate(secret_dataset, verbose=0)

    print(f"Additional Dataset Loss: {sec_loss:.4f}")
    print(f"Additional Dataset Accuracy: {sec_acc * 100:.2f}%")

else:
    print("No additional evaluation dataset found.")


## Residual CNN Model with Custom ResNet Blocks

To further improve feature extraction and model performance, a second architecture is built using **custom residual blocks inspired by ResNet**.

Instead of using a plain CNN stack, this model introduces **skip connections**, which help preserve information flow across layers and support deeper feature learning.

### Objective

The purpose of this experiment is to evaluate whether a residual architecture can match or improve upon the baseline CNN performance on the same food vs non-food classification task.

### What this section includes

- Implementation of custom residual blocks
- Construction of a small ResNet-style classifier
- Training on the same dataset and preprocessing pipeline
- Visualization of training and validation performance
- Saving the trained residual model for later evaluation


In [ ]:
# -------------------------------------------------------
# Residual Block Definition
# -------------------------------------------------------

def residual_block(x, filters, downsample=False, drop_rate=0.0):
    shortcut = x
    stride = 2 if downsample else 1

    # First convolution
    x = layers.Conv2D(
        filters, 3, strides=stride, padding="same", use_bias=False
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # Second convolution
    x = layers.Conv2D(
        filters, 3, padding="same", use_bias=False
    )(x)
    x = layers.BatchNormalization()(x)

    # Projection shortcut when dimensions change
    if downsample or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(
            filters, 1, strides=stride, padding="same", use_bias=False
        )(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    # Residual connection
    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)

    if drop_rate > 0:
        x = layers.Dropout(drop_rate)(x)

    return x


# -------------------------------------------------------
# Build ResNet-Style CNN Model
# -------------------------------------------------------

def make_resnet(drop_rate=0.3):
    inputs = keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))

    # Data augmentation + normalization
    x = data_augmentation(inputs)
    x = layers.Rescaling(1.0 / 255)(x)

    # Initial feature extraction layer
    x = layers.Conv2D(32, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # Residual stages
    x = residual_block(x, 32, downsample=False, drop_rate=drop_rate)
    x = residual_block(x, 32, downsample=False, drop_rate=drop_rate)

    x = residual_block(x, 64, downsample=True, drop_rate=drop_rate)
    x = residual_block(x, 64, downsample=False, drop_rate=drop_rate)

    x = residual_block(x, 128, downsample=True, drop_rate=drop_rate)
    x = residual_block(x, 128, downsample=False, drop_rate=drop_rate)

    # Classification head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)

    if drop_rate > 0:
        x = layers.Dropout(drop_rate)(x)

    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inputs, outputs, name="small_resnet_food_nonfood")

    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


# -------------------------------------------------------
# Train ResNet-Style Model
# -------------------------------------------------------

resnet_model = make_resnet(drop_rate=0.3)

history_R = resnet_model.fit(
    training_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    verbose=1
)


In [ ]:
# -------------------------------------------------------
# Plot ResNet Training Curves
# -------------------------------------------------------

plt.figure()
plt.plot(history_R.history["accuracy"], label="Training Accuracy")
plt.plot(history_R.history["val_accuracy"], label="Validation Accuracy")
plt.title("ResNet Model - Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure()
plt.plot(history_R.history["loss"], label="Training Loss")
plt.plot(history_R.history["val_loss"], label="Validation Loss")
plt.title("ResNet Model - Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()


# -------------------------------------------------------
# Save Trained ResNet Model
# -------------------------------------------------------

resnet_model.save("food_nonfood_resnet.keras")
print("Saved ResNet model: food_nonfood_resnet.keras")


## ResNet Model Evaluation

After training the residual CNN model, the saved ResNet architecture is loaded from disk and evaluated on the **Food-5K test dataset**.

The evaluation process measures how well the residual architecture generalizes to unseen data.

### Evaluation Steps

- Load the trained ResNet model
- Evaluate performance on the Food-5K test dataset
- Optionally evaluate on an additional dataset if available

This allows comparison between the **baseline CNN** and the **ResNet-based architecture**.


In [ ]:
# -------------------------------------------------------
# Load Saved ResNet Model
# -------------------------------------------------------

model = keras.models.load_model("food_nonfood_resnet.keras")


# -------------------------------------------------------
# Evaluate on Food-5K Test Dataset
# -------------------------------------------------------

test_dataset = make_eval_dataset(dataset_path / "evaluation")

test_loss, test_acc = model.evaluate(test_dataset, verbose=0)

print(f"ResNet Food-5K Test Loss: {test_loss:.4f}")
print(f"ResNet Food-5K Test Accuracy: {test_acc * 100:.2f}%")


# -------------------------------------------------------
# Optional Evaluation on Additional Dataset
# -------------------------------------------------------

if secret_dataset_path.exists():

    secret_dataset = make_eval_dataset(secret_dataset_path)

    sec_loss, sec_acc = model.evaluate(secret_dataset, verbose=0)

    print(f"Additional Dataset Loss: {sec_loss:.4f}")
    print(f"Additional Dataset Accuracy: {sec_acc * 100:.2f}%")

else:
    print("No additional evaluation dataset found.")


## Transfer Learning with EfficientNetB0

To further improve classification performance, a **pretrained convolutional neural network** is used as a feature extractor for the same food vs non-food classification task.

In this experiment, **EfficientNetB0** pretrained on **ImageNet** is used as the backbone model. The pretrained base is kept frozen, while a lightweight classification head is trained on the Food-5K dataset.

### Objective

The purpose of this experiment is to evaluate whether transfer learning can provide:

- Faster convergence
- Better feature extraction
- Improved validation and test performance compared to custom CNN architectures

### What this section includes

- Loading a pretrained EfficientNetB0 backbone
- Building a transfer-learning classifier
- Training the custom classification head
- Visualizing training and validation curves
- Saving the trained model for later evaluation


In [ ]:
# -------------------------------------------------------
# Build Transfer Learning Model with EfficientNetB0
# -------------------------------------------------------

base_model = keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)
)

# Freeze pretrained backbone
base_model.trainable = False

inputs = keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))

# Data augmentation + normalization
x = data_augmentation(inputs)
x = layers.Rescaling(1.0 / 255)(x)

# Feature extraction
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)

# Classification head
outputs = layers.Dense(1, activation="sigmoid")(x)

efficientnet_model = keras.Model(
    inputs,
    outputs,
    name="efficientnetb0_food_nonfood"
)

efficientnet_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Train classifier head
history_E = efficientnet_model.fit(
    training_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    verbose=1
)


In [ ]:
# -------------------------------------------------------
# Plot EfficientNetB0 Training Curves
# -------------------------------------------------------

plt.figure()
plt.plot(history_E.history["accuracy"], label="Training Accuracy")
plt.plot(history_E.history["val_accuracy"], label="Validation Accuracy")
plt.title("EfficientNetB0 - Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure()
plt.plot(history_E.history["loss"], label="Training Loss")
plt.plot(history_E.history["val_loss"], label="Validation Loss")
plt.title("EfficientNetB0 - Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()


# -------------------------------------------------------
# Save Trained EfficientNetB0 Model
# -------------------------------------------------------

efficientnet_model.save("food_nonfood_efficientnet.keras")
print("Saved EfficientNetB0 model: food_nonfood_efficientnet.keras")


## EfficientNetB0 Model Evaluation

After training, the saved **EfficientNetB0-based transfer learning model** is loaded from disk and evaluated on the **Food-5K test dataset**.

This evaluation measures how well the pretrained feature extractor generalizes to unseen data.

### Evaluation Steps

- Load the trained EfficientNetB0 model
- Evaluate on the Food-5K test dataset
- Optionally evaluate on an additional dataset if available

This enables direct comparison with the **baseline CNN** and **ResNet-based model**.


In [ ]:
# -------------------------------------------------------
# Load Saved EfficientNetB0 Model
# -------------------------------------------------------

model = keras.models.load_model("food_nonfood_efficientnet.keras")


# -------------------------------------------------------
# Evaluate on Food-5K Test Dataset
# -------------------------------------------------------

test_dataset = make_eval_dataset(dataset_path / "evaluation")

test_loss, test_acc = model.evaluate(test_dataset, verbose=0)

print(f"EfficientNetB0 Test Loss: {test_loss:.4f}")
print(f"EfficientNetB0 Test Accuracy: {test_acc * 100:.2f}%")


# -------------------------------------------------------
# Optional Evaluation on Additional Dataset
# -------------------------------------------------------

if secret_dataset_path.exists():

    secret_dataset = make_eval_dataset(secret_dataset_path)

    sec_loss, sec_acc = model.evaluate(secret_dataset, verbose=0)

    print(f"Additional Dataset Loss: {sec_loss:.4f}")
    print(f"Additional Dataset Accuracy: {sec_acc * 100:.2f}%")

else:
    print("No additional evaluation dataset found.")


## Final Conclusion

This project explored multiple deep learning approaches for **food vs non-food image classification** using the Food-5K dataset. Three different model architectures were implemented and evaluated:

1. **Custom CNN Model**  
   A baseline convolutional neural network was built using stacked convolution layers and pooling operations. This model established a reference performance for the classification task.

2. **Residual CNN (ResNet-style Model)**  
   A second architecture introduced **residual connections**, allowing deeper feature extraction while maintaining stable gradient flow. The residual blocks helped improve training stability and model capacity.

3. **Transfer Learning with EfficientNetB0**  
   A pretrained **EfficientNetB0** model was used as a feature extractor. Leveraging pretrained ImageNet representations significantly improved feature learning efficiency and allowed faster convergence compared to training from scratch.

### Key Insights

- **Data augmentation** helped improve generalization by exposing the model to more diverse input variations.  
- **Dropout regularization** reduced overfitting and stabilized validation performance.  
- **Residual connections** improved training stability for deeper networks.  
- **Transfer learning** provided the strongest feature extraction capability with minimal additional training.

### Overall Outcome

The comparison between these architectures demonstrates how increasingly advanced techniques—from basic CNNs to residual networks and transfer learning—can improve image classification performance.

### Future Work

Possible extensions of this project include:

- Fine-tuning the pretrained EfficientNet layers for further accuracy improvements  
- Training on larger food image datasets for better generalization  
- Deploying the trained model as a lightweight inference API or web application
